In [ ]:
import numpy as np

data = np.load("/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/patches/unified_test_normal.npz")

print(data.files)

In [ ]:
import numpy as np

data = np.load(
    "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/patches/unified_train_2d.npz",
    allow_pickle=True
)

scan_ids = data["scan_ids"]

unique_scans = np.unique(scan_ids)

print("Number of unique scans:", len(unique_scans))

for s in unique_scans:
    print(s)

In [ ]:
import numpy as np

data = np.load(
    "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/patches/unified_train_2d.npz",
    allow_pickle=True
)

scan_ids = data["scan_ids"]

unique_scans, counts = np.unique(
    scan_ids,
    return_counts=True
)

# Sort by least number of patches
least_idx = np.argsort(counts)

print("Scans with the fewest patches:\n")

for i in least_idx[:20]:   # show 20 least frequent scans
    print(f"{unique_scans[i]} : {counts[i]} patches")

# Zero Norms

In [ ]:
X = data["X"]

print("Patch matrix shape:", X.shape)

# Each column = one patch
patch_norms = np.linalg.norm(X, axis=0)

zero_mask = patch_norms < 1e-10


print("\n===== Patch Norm Statistics =====")
print("Total patches:", X.shape[1])
print("Zero norm patches:", zero_mask.sum())

print(
    f"Percentage zero norm patches: "
    f"{zero_mask.sum()/X.shape[1]*100:.6f}%"
)


print("\nNorm statistics")
print("Min norm:", patch_norms.min())
print("Max norm:", patch_norms.max())
print("Mean norm:", patch_norms.mean())
print("Median norm:", np.median(patch_norms))

In [ ]:
import numpy as np

patch_path = "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/patches/unified_train.npz"

data = np.load(
    patch_path,
    allow_pickle=True
)

X = data["X"]
scan_ids = data["scan_ids"]

# Compute norms
patch_norms = np.linalg.norm(X, axis=0)

# Find zero norm patches
zero_mask = patch_norms < 1e-10

# Get scan IDs corresponding to zero norm patches
zero_norm_scan_ids = scan_ids[zero_mask]


print("Number of zero norm patches:", len(zero_norm_scan_ids))

print("\nFirst 20 zero norm patch scan IDs:")
print(zero_norm_scan_ids[:20])

In [ ]:
unique_zero_scan_ids, counts = np.unique(
    zero_norm_scan_ids,
    return_counts=True
)

print("Number of scans containing zero-norm patches:",
      len(unique_zero_scan_ids))

for scan_id, count in zip(unique_zero_scan_ids, counts):
    print(scan_id, "->", count, "zero-norm patches")

# Train Normal Patches

In [ ]:
import numpy as np

data = np.load("/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/patches/unified_train_normal.npz", allow_pickle=True)

X = data["X"]

print("Shape:", X.shape)

print("NaN:", np.isnan(X).sum())
print("Inf:", np.isinf(X).sum())

print("Min:", X.min())
print("Max:", X.max())

In [ ]:
import numpy as np

path = "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/patches/unified_train_balanced.npz"

data = np.load(path, allow_pickle=True)

X = data["X"]
H = data["H"]
scan_ids = data["scan_ids"]


print("==============================")
print("Dataset shapes")

print("X:", X.shape)
print("H:", H.shape)
print("scan_ids:", scan_ids.shape)



print("\n==============================")
print("Class distribution")

classes, counts = np.unique(H, return_counts=True)

for c, n in zip(classes, counts):
    print(f"Class {c}: {n}")



print("\n==============================")
print("Unique CT scans")

unique_scans = np.unique(scan_ids)

print("Number of scans:", len(unique_scans))


print("\nFirst 10 scan IDs:")
for s in unique_scans[:10]:
    print(s)



print("\n==============================")
print("Patches per scan statistics")

patches_per_scan = []

for sid in unique_scans:
    patches_per_scan.append(
        np.sum(scan_ids == sid)
    )


patches_per_scan = np.array(patches_per_scan)


print("Min patches:", patches_per_scan.min())
print("Max patches:", patches_per_scan.max())
print("Mean patches:", patches_per_scan.mean())
print("Median patches:", np.median(patches_per_scan))



print("\n==============================")
print("Check alignment")

print(
    "X patches == labels:",
    X.shape[1] == len(H)
)

print(
    "Labels == scan ids:",
    len(H) == len(scan_ids)
)

# Find patch indices

In [ ]:
import numpy as np

data = np.load(
    "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/patches/unified_train_2d.npz",
    allow_pickle=True
)

X = data["X"]
scan_ids = data["scan_ids"]
coords = data["coords"]
H = data["H"]

target_scan = "train_11150_b_2"

# Find patch indices
indices = np.where(scan_ids == target_scan)[0]

print("Number of patches:", len(indices))
print("Patch matrix column indices:")
print(indices)

In [ ]:
coords = data["coords"]
scan_ids = data["scan_ids"]
scan_id = "train_11150_b_2"
indices = np.where(
    scan_ids == scan_id
)[0]

print("Indices:")
print(indices)

print("\nCoordinates:")
print(coords[indices])

# Visualize Patch Matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from lc_ksvd.data_loader.metadata_registry import MetadataRegistry
from lc_ksvd.data_loader.scan_loader import ScanLoader
from lc_ksvd.config import PATCH_SIZE, CLASS_ORDER

target_scan = "train_11150_b_2"
split = "train"   # matches the "train_" prefix / patch file split

# --- Load the actual preprocessed volume + resampled ground-truth mask ---
metadata = MetadataRegistry(split=split)
scan = ScanLoader(metadata).load(target_scan)

volume = scan["volume"]          # float32 [H, W, D] in [0, 1], resampled/windowed
mask = scan["mask"]              # uint8 [F, H, W, D] or None, same grid as volume
finding_map = scan["finding_map"]  # {f_idx: category}, e.g. {0: "2c"}

print("Volume shape:", volume.shape)
print("Finding map:", finding_map)

# --- Patch coords for this scan (already in the same coordinate space as volume) ---
patch_data = np.load(
    "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/patches/unified_train_2d.npz",
    allow_pickle=True,
)
scan_ids = patch_data["scan_ids"]
coords = patch_data["coords"]

indices = np.where(scan_ids == target_scan)[0]
scan_coords = coords[indices]

# --- Collapse ground truth mask (any finding) into a single binary volume ---
gt = None
if mask is not None:
    gt = mask.any(axis=0)  # [H, W, D] bool, union over all findings

# --- Pick a slice to visualize: center of the patch cloud along z ---
if len(scan_coords) > 0:
    cz = int(np.median(scan_coords[:, 2] + PATCH_SIZE // 2))
else:
    cz = volume.shape[2] // 2

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(volume[:, :, cz], cmap="gray")

# Overlay ground truth mask (red, semi-transparent)
if gt is not None:
    gt_slice = gt[:, :, cz]
    overlay = np.zeros((*gt_slice.shape, 4))
    overlay[gt_slice] = [1, 0, 0, 0.4]  # red, alpha 0.4
    ax.imshow(overlay)

# Overlay patch boxes that intersect this slice
for (x0, y0, z0) in scan_coords:
    if z0 <= cz < z0 + PATCH_SIZE:
        rect = mpatches.Rectangle(
            (y0, x0), PATCH_SIZE, PATCH_SIZE,
            linewidth=1, edgecolor="lime", facecolor="none",
        )
        ax.add_patch(rect)

ax.set_title(f"{target_scan} — slice z={cz} (green=patches, red=GT)")
ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.hist(volume.flatten(), bins=100)
plt.xlabel("Reconstructed intensity")
plt.ylabel("Voxel count")
plt.title("Reconstructed Volume Histogram")
plt.show()

print("min:", volume.min())
print("max:", volume.max())
print("mean:", volume.mean())
print("std:", volume.std())

In [ ]:
# choose threshold
threshold = np.percentile(volume[volume != 0], 50)
print("Threshold:", threshold)

segmentation = volume > threshold
print("Foreground voxels:", segmentation.sum())

In [ ]:
cx, cy, cz = np.array(segmentation.shape)//2

fig, ax = plt.subplots(1,3,figsize=(12,4))

ax[0].imshow(segmentation[:,:,cz], cmap="gray")
ax[0].set_title("Axial")

ax[1].imshow(segmentation[:,cy,:], cmap="gray")
ax[1].set_title("Coronal")

ax[2].imshow(segmentation[cx,:,:], cmap="gray")
ax[2].set_title("Sagittal")

for a in ax:
    a.axis("off")

plt.show()

## Otsu threshold

In [ ]:
from skimage.filters import threshold_otsu

nonzero = volume[volume != 0]

threshold = threshold_otsu(nonzero)
print("Otsu threshold:", threshold)

segmentation = volume > threshold

In [ ]:
from scipy.ndimage import binary_opening, binary_closing, label
from scipy.ndimage import label

segmentation = binary_opening(
    segmentation,
    iterations=1
)

segmentation = binary_closing(
    segmentation,
    iterations=2
)

labeled, num = label(segmentation)

sizes = np.bincount(labeled.ravel())

# remove background
sizes[0] = 0

largest_component = sizes.argmax()

segmentation = labeled == largest_component

print(
    "Final segmentation voxels:",
    segmentation.sum()
)

In [ ]:
cx,cy,cz = np.array(volume.shape)//2

fig,ax = plt.subplots(1,3,figsize=(12,4))

ax[0].imshow(volume[:,:,cz], cmap="gray")
ax[0].contour(segmentation[:,:,cz], colors="red")
ax[0].set_title("Axial")

ax[1].imshow(volume[:,cy,:], cmap="gray")
ax[1].contour(segmentation[:,cy,:], colors="red")
ax[1].set_title("Coronal")

ax[2].imshow(volume[cx,:,:], cmap="gray")
ax[2].contour(segmentation[cx,:,:], colors="red")
ax[2].set_title("Sagittal")

for a in ax:
    a.axis("off")

plt.show()

In [ ]:
import nibabel as nib
import os

# Convert boolean mask to uint8
segmentation_uint8 = segmentation.astype(np.uint16)

# Save path
save_path = f"/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/reconstructed/{scan_id}.nii.gz"

# Create identity affine (because this is a cropped reconstruction)
affine = np.eye(4)

# Create NIfTI image
nii = nib.Nifti1Image(
    segmentation_uint8,
    affine
)

# Save
nib.save(
    nii,
    save_path
)

print("Saved segmentation:")
print(save_path)

print("Shape:", segmentation_uint8.shape)
print("Foreground voxels:", segmentation_uint8.sum())